# 🧰 Sesión Extra: Herramientas prácticas para proyectos NLP en Python
**Temas:** Poetry, Ortografía, STT/TTS, Web Scraping


## 🎯 Objetivos de la sesión
- Gestionar ambientes con **Poetry**
- Implementar **corrección ortográfica** en Python
- Usar **Speech-to-Text (STT)** y **Text-to-Speech (TTS)**
- Realizar **Web Scraping** con librerías fundamentales


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1️⃣ Ambientes virtuales y gestor de paquetes (Poetry)

In [ ]:

# Instalación (ejecutar en terminal, no en notebook)
# curl -sSL https://install.python-poetry.org | python3
# poetry --version


In [ ]:

# Flujo típico con Poetry (terminal)
# poetry new mi_proyecto
# cd mi_proyecto
# poetry add spacy pyspellchecker requests beautifulsoup4 lxml spellchecker unidecode language_tool_python
# poetry shell

# poetry env activate

# poetry lock


## 2️⃣ Ortografía en Python 📝

In [ ]:
pip install spellchecker unidecode language_tool_python

In [ ]:
!pip install pyspellchecker

In [ ]:
import pandas as pd

from spellchecker import SpellChecker
from unidecode import unidecode
import re
from typing import List, Tuple, Dict

# (Opcional) LanguageTool para gramática:
import language_tool_python

In [ ]:
# Utilidades para dividir en "palabras" manteniendo signos
TOKEN_RE = re.compile(r"(\w+|[¿?¡!.,;:\-—\(\)\[\]\{\}\"'«»…]+|\s+)", re.UNICODE)

def tokenize_mixed(text: str) -> List[str]:
    """
    Tokenize the input text while preserving punctuation and spaces.

    Args:
        text (str): The input text to be tokenized.

    Returns:
        List[str]: A list of tokens where words, punctuation marks, and whitespace
        are preserved as separate elements.
    """
    return [t for t in TOKEN_RE.findall(text) if t != ""]


def is_word(token: str) -> bool:
    """
    Determine whether a token should be considered a word.

    Args:
        token (str): The token to evaluate.

    Returns:
        bool: True if the token is non-empty and alphanumeric;
        False otherwise.
    """
    return token.strip() != "" and token.strip().isalnum()
    #t = token.strip()
    #return t != "" and t.isalpha()


def restore_casing(original: str, corrected: str) -> str:
    """
    Adjust the casing of a corrected word to match the casing pattern of the original word.

    Args:
        original (str): The original token with its original casing.
        corrected (str): The corrected version of the token.

    Returns:
        str: The corrected token with casing adapted to follow the original pattern
        (uppercase, title case, or lowercase). If mixed case, the corrected form is returned as-is.
    """
    if original.isupper():
        return corrected.upper()
    if original.istitle():
        return corrected.capitalize()
    if original.islower():
        return corrected.lower()
    # Mixed case: keep the suggestion as-is
    return corrected


In [ ]:
spell = SpellChecker(language="es")
texto = "El raton se esacpo de la csa"
palabras = texto.split()
correcciones = {p: spell.correction(p) for p in palabras if p not in spell}
correcciones


{'raton': 'ratón', 'esacpo': 'escapo', 'csa': 'casa'}

## 🔁 Corrección ortográfica básica
La idea es:
1. Tokenizar el texto preservando puntuación.
2. Para cada palabra, si no está en el diccionario, sugerir una corrección.
3. Reconstruir el texto con las correcciones.

### ¿Qué es un `@dataclass`?

En Python, un **dataclass** (disponible desde la versión 3.7) es una clase de datos que se define con el decorador `@dataclass`.  
Este decorador permite crear clases pensadas principalmente para **almacenar información**, sin necesidad de escribir código repetitivo.

Al aplicar `@dataclass`, Python genera automáticamente métodos especiales como:
- `__init__`: el constructor de la clase, con todos los atributos.
- `__repr__`: una representación legible del objeto.
- `__eq__`: comparación de igualdad entre instancias.

Su objetivo es simplificar el manejo de objetos que solo contienen datos.  
En este caso, la clase `Correction` nos permite guardar de manera ordenada:
- la palabra original,
- la sugerencia de corrección,
- las posibles alternativas,
- y un indicador de si se reemplazó o no.

In [ ]:
from dataclasses import dataclass

@dataclass
class Correction:
    """
    Data structure representing a single spelling correction.

    Attributes:
        original (str): The original token as it appeared in the input text.
        suggestion (str): The corrected token (may be identical to the original if no correction was applied).
        candidates (List[str]): A list of candidate corrections suggested by the spell checker.
        replaced (bool): True if the original token was replaced by a suggestion; False otherwise.
    """
    original: str
    suggestion: str
    candidates: List[str]
    replaced: bool

def spell_correct_text(text: str, ignore_case_accents: bool = True) -> Tuple[str, List[Correction]]:
    """
    Perform spell checking and correction on Spanish text.

    The function tokenizes the input text, detects misspelled words, and replaces them with the most probable
    correction while preserving punctuation, spacing, and casing.

    Args:
        text (str): The input text to check and correct.
        ignore_case_accents (bool, optional): If True, accent marks and case are ignored when checking spelling.
                                              Defaults to True.

    Returns:
        Tuple[str, List[Correction]]:
            - str: The corrected text reconstructed with suggested replacements.
            - List[Correction]: A list of Correction objects containing details about each token
                                (original word, suggestion, candidates, and replacement flag).
    """
    tokens = tokenize_mixed(text)
    corrections = []
    new_tokens = []

    # Pre-cálculo de palabras mal escritas (case-insensitive con opción de quitar acentos)
    check_tokens = []
    mapping = []
    for tok in tokens:
        if is_word(tok):
            key = tok.lower()
            if ignore_case_accents:
                key = unidecode(key)  # quita tildes para la consulta
            check_tokens.append(key)
            mapping.append(tok)
        else:
            check_tokens.append(None)
            mapping.append(tok)

    misspelled = set()
    for i, key in enumerate(check_tokens):
        if key is None:
            continue
        # Consideramos "bien escrita" si existe tal cual en el diccionario.
        # Probamos con y sin acentos.
        if key not in spell and unidecode(key) not in spell:
            misspelled.add(i)

    for i, tok in enumerate(tokens):
        if i not in misspelled or not is_word(tok):
            new_tokens.append(tok)
            if is_word(tok):
                corrections.append(Correction(tok, tok, [], False))
            continue

        # Sugerimos corrección usando la forma sin acentos ni mayúsculas
        key = tok.lower()
        if ignore_case_accents:
            key = unidecode(key)

        key = re.sub(r"[^a-záéíóúüñ]", "", key, flags=re.IGNORECASE)
        if not key:
            # No intentamos corregir tokens vacíos/raros
            new_tokens.append(tok)
            corrections.append(Correction(tok, tok, [], False))
            continue

        # Candidatos
        cset = spell.candidates(key) or set()        # fallback si devuelve None
        candidates = list(cset)
        suggestion = spell.correction(key) or tok     # si no hay sugerencia, mantener el original


        if suggestion is None:
            new_tokens.append(tok)
            corrections.append(Correction(tok, tok, [], False))
        else:
            # Restaurar mayúsculas/minúsculas
            fixed = restore_casing(tok, suggestion)
            new_tokens.append(fixed)
            corrections.append(Correction(tok, fixed, candidates, fixed.lower() != tok.lower()))

    # Unir tokens cuidando espacios
    corrected_text = "".join(new_tokens)
    return corrected_text, corrections

In [ ]:
sample_text = "El proffesor me dio las instruciones para el exámen de maána. Tambíen dijo que traigamos lapiz."
corrected, details = spell_correct_text(sample_text)
print("TEXTO ORIGINAL:")
print(sample_text)
print("\nTEXTO CORREGIDO:")
print(corrected)


TEXTO ORIGINAL:
El proffesor me dio las instruciones para el exámen de maána. Tambíen dijo que traigamos lapiz.

TEXTO CORREGIDO:
El profesor me dios las instruciones para el exámen de mañana. También hijo que traigamos lápiz.


In [ ]:
# Mostrar detalle
df = pd.DataFrame([{
    "original": c.original,
    "sugerencia": c.suggestion,
    "candidatos": ", ".join(c.candidates)[:120] + ("..." if len(", ".join(c.candidates))>120 else ""),
    "reemplazó?": c.replaced
} for c in details])
df

,original,sugerencia,candidatos,reemplazó?
0,El,El,,False
1,proffesor,profesor,profesor,True
2,me,me,,False
3,dio,dios,"dios, di, din, dito, dúo, dino, do, diz, divo,...",True
4,las,las,,False
5,instruciones,instruciones,,False
6,para,para,,False
7,el,el,,False
8,exámen,exámen,,False
9,de,de,,False


### Notas
- `pyspellchecker` usa frecuencias de palabras; **no entiende contexto** (p. ej., *cazar* vs *casar*).
- Para textos con **nombres propios** o jerga, añade palabras al diccionario con `spell.word_frequency.add('MiMarca')`.
- Puedes desactivar `ignore_case_accents=False` si deseas estrictitud con tildes.[

## Otras opciones:

### LanguageTool (solo) — ortografía + gramática

In [ ]:
!apt-get -y install openjdk-17-jre-headless

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jre-headless is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!pip install language-tool-python unidecode

In [ ]:
import language_tool_python as lt

tool = lt.LanguageTool('es')  # 'es-ES'
texto = "El raton se esacpo de la csa, y los datos esta mal escritos."

matches = tool.check(texto)
corregido = lt.utils.correct(texto, matches)

In [ ]:
print("Original:", texto)
print("Corregido:", corregido)
for m in matches[:5]:
    print("-", m.rule_id, m.message, f"[{m.replacements[:3]}]")

Original: El raton se esacpo de la csa, y los datos esta mal escritos.
Corregido: El ratón se escapó de la casa, y los datos está mal escritos.
- MORFOLOGIK_RULE_ES Se ha encontrado un posible error ortográfico. [['ratón', 'razón', 'Ramón']]
- MORFOLOGIK_RULE_ES Se ha encontrado un posible error ortográfico. [['escapó', 'escapo']]
- MORFOLOGIK_RULE_ES Se ha encontrado un posible error ortográfico. [['casa', 'esa', 'usa']]
- ESTA_TILDE Si es del verbo ‘estar’, se escribe con tilde. [['está']]


### SymSpell (candidatos rápidos) + lista blanca

In [ ]:
!pip install symspellpy unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 11.5 MB/s eta 0:00:00


In [ ]:
from symspellpy import SymSpell, Verbosity
from unidecode import unidecode

In [ ]:
sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
# Diccionario (palabra frecuencia). Puedes crear el tuyo con corpus propio:
# sym.load_dictionary("es_dictionary.txt", term_index=0, count_index=1)
# O bootstrap mínimo:
for w in ["ratón", "casa", "escapó", "Bogotá", "NLP", "datos"]:
    sym.create_dictionary_entry(w, 100000)

whitelist = {"NLP","Bogotá"}  # no corregir

def suggest_sym(text: str):
    out = []
    for tok in text.split():
        if tok in whitelist or not tok.isalpha():
            out.append(tok); continue
        sugg = sym.lookup(tok, Verbosity.CLOSEST, max_edit_distance=2)
        out.append(sugg[0].term if sugg else tok)
    return " ".join(out)

In [ ]:
texto = "El raton se esacpo de la csa en Bogota para NLP."
print(suggest_sym(texto))

El ratón se escapó de la casa en Bogotá casa NLP.


### Pipeline híbrido sencillo (SymSpell → LanguageTool)

In [ ]:
def corregir(texto: str) -> str:
    # 1) candidatos rápidos
    pre = suggest_sym(texto)
    # 2) gramática/acentos/concordancia
    matches = tool.check(pre)
    return lt.utils.correct(pre, matches)

In [ ]:
print(corregir("El raton se esacpo de la csa, y los dato estan mal."))

El ratón se escapó de la casa, y los datos están mal.


### Consejos de calidad (específico para español)

Acentos y mayúsculas: normaliza y luego restaura capitalización de nombres propios.

Tokenización robusta: ignora URLs, emails, números y siglas en mayúsculas.

Diccionarios de dominio: añade glosarios (médico, legal) al motor elegido.

Lista blanca/negra: evita sobrecorrecciones (“modelado”, “tokenización”).

Evaluación: guarda pares (original, corregido) y mide precisión/recall sobre un conjunto etiquetado por ti.

## 3️⃣ Speech-to-Text y Text-to-Speech 🎙️🔊

In [ ]:
# Instalación (ejecutar si es necesario)
!pip install SpeechRecognition gTTS pyaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)


In [ ]:
!pip install pyttsx3 gTTS SpeechRecognition pydub sounddevice scipy vosk

  Using cached gTTS-2.5.4-py3-none-any.whl.metadata (4.1 kB)
  Using cached speechrecognition-3.16.1-py3-none-any.whl.metadata (28 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Preparing metadata (setup.py) ... done
Using cached gTTS-2.5.4-py3-none-any.whl (29 kB)
Using cached speechrecognition-3.16.1-py3-none-any.whl (32.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 38.2 MB/s eta 0:00:00
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=d81590d5b46e78360e2e1fe826c9b16d15c52f3fcd3e7a9062ff9677ac89243e
  Stored in directory: /root/.cache/pip/wheels/7e/75/5b/e1d5c3756631e4bda806f6cc9640153b39484bb6f7b0b8def3
Successfully built srt
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [ ]:
!apt-get update -y
!apt-get install -y espeak-ng

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,990 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,862 kB]
Get:13 http://security.ubuntu.com/ubun

**Funciona en local!**

In [ ]:
import pyttsx3

In [ ]:
def tts_offline_pyttsx3(text: str, rate: int = 180, volume: float = 1.0, voice_contains: str = "spanish"):
    """
    Synthesize speech from text using the offline pyttsx3 engine.

    This function runs entirely offline by leveraging the system's installed voices.
    It attempts to select a Spanish voice if `voice_contains` is found in the voice name/ID;
    otherwise, it falls back to the default system voice.

    Args:
        text (str): The input text to be spoken.
        rate (int, optional): Speaking rate in words per minute (WPM). Defaults to 180.
        volume (float, optional): Output volume in the range [0.0, 1.0]. Defaults to 1.0.
        voice_contains (str, optional): Case-insensitive substring used to select a specific voice
                                        (e.g., "spanish" or "es"). Defaults to "spanish".

    Returns:
        None: The function plays audio directly through the system audio device.
    """
    engine = pyttsx3.init()
    engine.setProperty('rate', rate)
    engine.setProperty('volume', volume)
    # Seleccionar una voz que contenga 'spanish' o 'es'
    selected = None
    for v in engine.getProperty('voices'):
        name = f"{v.name} ({v.id})".lower()
        #if (voice_contains.lower() in name or "es" in name) and (v.id != "HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_EN-US_DAVID_11.0"):
        if voice_contains.lower() in name or "es" in name:
            engine.setProperty('voice', v.id)
            selected = v.id
            break
    if selected is None:
        print("No se encontró voz en español; se usará la predeterminada.")
    engine.say(text)
    engine.runAndWait()

In [ ]:
tts_offline_pyttsx3("Hola, esto es una prueba de síntesis de voz en español. ¡Funciona sin internet!")

Exception ignored on calling ctypes callback function: <bound method EspeakDriver._onSynth of <pyttsx3.drivers.espeak.EspeakDriver object at 0x7cd11a25ffb0>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyttsx3/drivers/espeak.py", line 245, in _onSynth
    self._proxy.notify("finished-utterance", completed=True)
    ^^^^^^^^^^^^^^^^^^
ReferenceError: weakly-referenced object no longer exists


⚠️ Problema: Colab no puede reproducir audio directamente en la notebook con pyttsx3, aunque se instale eSpeak. El motor generará sonido en el contenedor (que no llega a tu navegador).

**Ajuste para que se almacene el archivo de audio**

In [ ]:
import pyttsx3
from pathlib import Path
from IPython.display import Audio

In [ ]:
def tts_pyttsx3_to_file(text: str, path: str = "salida.wav",
                        rate: int = 180, volume: float = 1.0,
                        voice_contains: str = "spanish") -> str:
    """
    Genera un archivo de audio offline usando pyttsx3 (motor TTS local).
    Funciona si hay un motor instalado en el sistema (espeak, sapi, etc.).
    """
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)
    engine.setProperty("volume", volume)

    # Seleccionar voz en español si existe
    for v in engine.getProperty("voices"):
        name = f"{v.name} ({v.id})".lower()
        if voice_contains.lower() in name or "es" in name:
            engine.setProperty("voice", v.id)
            break

    engine.save_to_file(text, path)
    engine.runAndWait()
    return path

In [ ]:
# Generar archivo
ruta = tts_pyttsx3_to_file("Hola, este es un ejemplo guardado con pyttsx3.", "demo.wav")
Audio(ruta)  # para reproducir en Jupyter/Colab

Audio saved to demo.wav


## 🔤➡️🔉 (🌐) TTS Online con `gTTS`
Genera audio con buena calidad; necesita conexión. Guardamos y reproducimos en el notebook.

In [ ]:
from gtts import gTTS
from IPython.display import Audio
from pathlib import Path

In [ ]:
def tts_gtts(text: str, lang: str = 'es', outfile: str = 'salida_tts_gtts.mp3'):
    """
    Generate speech audio from text using Google Text-to-Speech (gTTS).

    This function requires an internet connection. It creates an MP3 file with the synthesized
    speech and returns the output path.

    Args:
        text (str): The input text to synthesize.
        lang (str, optional): Language code for TTS (e.g., 'es', 'es-us', 'es-es'). Defaults to 'es'.
        outfile (str, optional): Output MP3 filename or path. Defaults to 'salida_tts_gtts.mp3'.

    Returns:
        str: The path to the generated MP3 file.
    """
    tts = gTTS(text=text, lang=lang)
    tts.save(outfile)
    return outfile

In [ ]:
mp3_path = tts_gtts("Hola, esta es una prueba con g T T S en español latino.", lang="es")
Audio(filename=mp3_path, autoplay=False)

### Usando OpenAI

```Python
!pip install openai

from openai import OpenAI
from pathlib import Path
from IPython.display import Audio

client = OpenAI(api_key="sk-")

# Generar audio con el modelo tts-1
speech_file = Path("openai_tts.mp3")
with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="alloy",   # otras voces: "verse", "sage"...
    input="Hola, este es un ejemplo con OpenAI TTS en español."
) as response:
    response.stream_to_file(speech_file)

Audio("openai_tts.mp3")

```

## 🔉➡️🔤 (🌐) STT Online con `SpeechRecognition` (Google Web Speech)
Muy sencillo, pero requiere internet. Puedes usar un archivo WAV/FLAC u OGG.

In [ ]:
import speech_recognition as sr

In [ ]:
def stt_online_google(audio_path: str, language: str = "es-ES") -> str:
    """
    Transcribe speech to text using SpeechRecognition with Google Web Speech API.

    This method processes a local audio file (WAV/FLAC/OGG) and requires an internet connection.
    It is simple to use but may be subject to service limits and network variability.

    Args:
        audio_path (str): Path to the input audio file.
        language (str, optional): BCP-47 language tag (e.g., 'es-ES', 'es-MX'). Defaults to 'es-ES'.

    Returns:
        str: The recognized text transcription.

    Raises:
        speech_recognition.UnknownValueError: If the speech is unintelligible.
        speech_recognition.RequestError: If the API request fails (e.g., connectivity issues).
    """
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = r.record(source)
    # Usa el servicio gratuito (límite de uso)
    return r.recognize_google(audio, language=language)


In [ ]:
# EJEMPLO (ajusta la ruta del modelo y del wav):
texto = stt_online_google("/content/drive/MyDrive/2025-2/NLP/Grabacion.wav", language="es-CO")
print(texto)

Este es un audio de prueba


## 🔉➡️🔤 (💻) STT Offline con `vosk`
Necesitas descargar un modelo de Vosk en español (por ejemplo: `vosk-model-small-es-0.42`).
1. Descarga y descomprime el modelo desde: https://alphacephei.com/vosk/models (elige español).
2. Ajusta `MODEL_DIR` a la carpeta del modelo.
O también
```Python
!wget -q -O /content/vosk-model-small-es-0.42.zip \
  https://alphacephei.com/vosk/models/vosk-model-small-es-0.42.zip
!unzip -q /content/vosk-model-small-es-0.42.zip -d /content/
```

In [ ]:
import json, wave, os
from vosk import Model, KaldiRecognizer

def stt_offline_vosk(audio_wav_path: str, model_dir: str) -> str:
    """
    Transcribe speech to text offline using Vosk.

    The input must be a mono, 16-bit PCM WAV file with a standard sampling rate.
    A valid Spanish Vosk model directory is required (download separately).

    Args:
        audio_wav_path (str): Path to a mono 16-bit PCM WAV file (e.g., 16 kHz).
        model_dir (str): Directory containing the Vosk model files (e.g., 'vosk-model-small-es-0.42').

    Returns:
        str: The recognized text transcription (may be empty if no speech is detected).

    Raises:
        FileNotFoundError: If the model directory does not exist.
        ValueError: If the WAV format is not mono 16-bit PCM or the sampling rate is unsupported.
    """
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"No existe el directorio del modelo: {model_dir}")
    model = Model(model_dir)
    wf = wave.open(audio_wav_path, "rb")
    if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getframerate() not in [8000, 16000, 32000, 44100, 48000]:
        raise ValueError("El WAV debe ser mono, 16-bit. Reconvierte si es necesario.")
    rec = KaldiRecognizer(model, wf.getframerate())
    rec.SetWords(True)
    results = []
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            results.append(json.loads(rec.Result()))
    results.append(json.loads(rec.FinalResult()))
    # Concatenar texto
    text = " ".join([r.get("text", "") for r in results]).strip()
    return text

In [ ]:
# EJEMPLO (ajusta la ruta del modelo y del wav):
texto = stt_offline_vosk(r"/content/drive/MyDrive/2025-2/NLP/Grabacion.wav", model_dir="/content/drive/MyDrive/2025-2/NLP/vosk-model-small-es-0.42")
print(texto)

este es un audio de prueba


## 🎤➡️🔤 (💻) Grabación rápida desde micrófono (opcional)
Para mayor control y evitar problemas de instalación de `pyaudio`, usaremos `sounddevice` para grabar y guardaremos a WAV.

#### Esto funciona el local

In [ ]:
!pip install sounddevice

In [ ]:
import sounddevice as sd

OSError: PortAudio library not found

In [ ]:
import numpy as np
from scipy.io.wavfile import write as wavwrite
from datetime import datetime

In [ ]:
def grabar_wav(segundos: int = 5, samplerate: int = 16000, outfile: str | None = None) -> str:
    """
    Record microphone audio and save it as a mono 16-bit PCM WAV file.

    This utility uses `sounddevice` for cross-platform recording and writes the result using `scipy`.
    The output filename is autogenerated if not provided.

    Args:
        segundos (int, optional): Recording duration in seconds. Defaults to 5.
        samplerate (int, optional): Sampling rate in Hz (e.g., 16000, 44100). Defaults to 16000.
        outfile (str | None, optional): Output filename; if None, a timestamped name is used. Defaults to None.

    Returns:
        str: The path to the recorded WAV file.

    Notes:
        - Ensure your environment has microphone access and the correct input device selected.
        - For best STT results (e.g., Vosk), use mono 16-bit PCM at 16 kHz in a quiet environment.
    """
    if outfile is None:
        outfile = f"grabacion_{datetime.now().strftime('%Y%m%d_%H%M%S')}.wav"
    print(f"Grabando {segundos} segundos...")
    audio = sd.rec(int(segundos * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait()
    wavwrite(outfile, samplerate, audio)
    print("Guardado en:", outfile)
    return outfile

In [ ]:
# EJEMPLO:
path = grabar_wav(4)
print(stt_offline_vosk(path, model_dir="vosk-model-small-es-0.42"))
print(stt_online_google(path, language="es-ES"))

#### Esto funciona el colab

In [ ]:
import io
import base64
from IPython.display import display, Javascript
from google.colab import output
import numpy as np
import soundfile as sf

In [ ]:
def record(sec=5, filename='grabacion.wav'):
    js = Javascript(f"""
    async function record(sec) {{
      const stream = await navigator.mediaDevices.getUserMedia({{audio:true}});
      const recorder = new MediaRecorder(stream);
      let data = [];
      recorder.ondataavailable = e => data.push(e.data);
      recorder.start();
      await new Promise(r => setTimeout(r, sec*1000));
      recorder.stop();
      await new Promise(r => recorder.onstop = r);
      const blob = new Blob(data);
      const arrayBuffer = await blob.arrayBuffer();
      const b64 = btoa(String.fromCharCode(...new Uint8Array(arrayBuffer)));
      google.colab.kernel.invokeFunction('notebook.record', [b64], {{}});
    }}
    record({sec});
    """)
    display(js)

    def _record(b64):
        audio = base64.b64decode(b64)
        with open(filename, 'wb') as f:
            f.write(audio)
    output.register_callback('notebook.record', _record)


In [ ]:
record(4, "grabacion.wav")

<IPython.core.display.Javascript object>

In [ ]:
!apt-get -y update >/dev/null
!apt-get -y install ffmpeg >/dev/null

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from pathlib import Path
import subprocess, os

In [ ]:
SRC = "/content/grabacion.wav"  # el archivo que generaste con JS
DST = "/content/grabacion16k.wav"

# Inspección rápida (opcional)
!ffprobe -hide_banner -loglevel warning -show_streams -select_streams a "{SRC}"

# Conversión robusta a PCM s16le, mono, 16 kHz
cmd = [
    "ffmpeg", "-y", "-i", SRC,
    "-ac", "1", "-ar", "16000",
    "-c:a", "pcm_s16le",
    DST
]
subprocess.run(cmd, check=True)
print("Convertido a WAV correcto:", Path(DST).exists(), DST)


[STREAM]
index=0
codec_name=opus
codec_long_name=Opus (Opus Interactive Audio Codec)
profile=unknown
codec_type=audio
codec_tag_string=[0][0][0][0]
codec_tag=0x0000
sample_fmt=fltp
sample_rate=48000
channels=1
channel_layout=mono
bits_per_sample=0
id=N/A
r_frame_rate=0/0
avg_frame_rate=0/0
time_base=1/1000
start_pts=0
start_time=0.000000
duration_ts=N/A
duration=N/A
bit_rate=N/A
max_bit_rate=N/A
bits_per_raw_sample=N/A
nb_frames=N/A
nb_read_frames=N/A
nb_read_packets=N/A
DISPOSITION:default=1
DISPOSITION:dub=0
DISPOSITION:original=0
DISPOSITION:comment=0
DISPOSITION:lyrics=0
DISPOSITION:karaoke=0
DISPOSITION:forced=0
DISPOSITION:hearing_impaired=0
DISPOSITION:visual_impaired=0
DISPOSITION:clean_effects=0
DISPOSITION:attached_pic=0
DISPOSITION:timed_thumbnails=0
TAG:language=eng
[/STREAM]
Convertido a WAV correcto: True /content/grabacion16k.wav


In [ ]:
!pip install -q pydub soundfile

In [ ]:
from pydub import AudioSegment

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
src = AudioSegment.from_file(SRC)          # detecta webm/ogg/mp3
src = src.set_channels(1).set_frame_rate(16000)
src.export(DST, format="wav")              # WAV PCM por defecto

<_io.BufferedRandom name='/content/grabacion16k.wav'>

In [ ]:
!pip install -q vosk soundfile

In [ ]:
import vosk, soundfile as sf, numpy as np, json

MODEL_DIR = "/content/drive/MyDrive/2025-2/NLP/vosk-model-small-es-0.42"  # tu ruta en Drive

In [ ]:
def stt_offline_vosk(audio_path: str, model_dir: str) -> str:
    model = vosk.Model(model_dir)
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    rec = vosk.KaldiRecognizer(model, sr); rec.SetWords(True)
    parts, step = [], 4000
    for i in range(0, len(audio), step):
        chunk = (np.array(audio[i:i+step] * 32767, dtype=np.int16)).tobytes()
        if rec.AcceptWaveform(chunk):
            parts.append(json.loads(rec.Result()).get("text", ""))
    parts.append(json.loads(rec.FinalResult()).get("text", ""))
    return " ".join(t for t in parts if t).strip()

In [ ]:
texto_vosk = stt_offline_vosk(DST, MODEL_DIR)
print("Vosk STT:", texto_vosk if texto_vosk else "(vacío)")

Vosk STT: la hola uno dos tres cuatro cinco seis


## 4️⃣ Web Scraping 🌐🕷️

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup, Comment
from urllib.request import urlopen

## Cómo hacer con páginas que tienen la información en HTML

In [ ]:
url = "https://fbref.com/en/comps/9/Premier-League-Stats"
html = urlopen(url)
html

HTTPError: HTTP Error 403: Forbidden

In [ ]:
soup = BeautifulSoup(html, 'lxml')
type(soup)

In [ ]:
# Get the title
title = soup.title
print(title)

In [ ]:
# Print out the text
text = soup.get_text()
print(soup.text)

In [ ]:
soup.find_all('a')

In [ ]:
all_links = soup.find_all("a")
for link in all_links:
    print(link.get("href"))


In [ ]:
tables = soup.find_all('table')

In [ ]:
tables[0]

In [ ]:
table = soup.select_one('table')

In [ ]:
table

In [ ]:
rows = table.find_all('tr')

In [ ]:
rows[0]

In [ ]:
rows[1]

In [ ]:
table.select("thead th")

In [ ]:
ths = table.select("thead th")

In [ ]:
ths

In [ ]:
ths[0].get_text(strip=True)

In [ ]:
headers = [th.get_text(strip=True) for th in table.select("thead th")]
headers

In [ ]:
rows[0].find_all('td')

In [ ]:
rows[1].find_all('td')

In [ ]:
type(rows[1].find_all('td'))

In [ ]:
table.select("tbody tr")

In [ ]:
table.select("tbody tr")[0]

In [ ]:
rows = []
for tr in table.select("tbody tr"):
    cells = [c.get_text(strip=True) for c in tr.select("th, td")]
    if cells:
        rows.append(cells)

In [ ]:
rows

In [ ]:
df = pd.DataFrame(rows, columns=headers[:len(rows[0])])
df.head()

In [ ]:
df.info()

In [ ]:
numeric_cols = ['MP', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts', 'Pts/MP','xG', 'xGA', 'xGD', 'xGD/90', 'Attendance']

In [ ]:
df[numeric_cols]

In [ ]:
df.GD[19]

In [ ]:
def to_num(s):
    if s is None:
        return None
    s = s.replace(",", "")      # 30,845 -> 30845
    s = s.replace("+", "")      # 17% -> 17
    #s = s.replace("\u2212", "-")# signo menos unicode
    s = s.strip()
    return pd.to_numeric(s, errors="ignore")


In [ ]:
for col in numeric_cols:
    df[col] = df[col].map(to_num)

In [ ]:
df[numeric_cols]

In [ ]:
df[numeric_cols].info()

In [ ]:
tables_pd = pd.read_html(url)         # o pd.read_html(url)
df = tables_pd[0]

In [ ]:
df.info()

In [ ]:
df

## Hagamos otro ensayo

In [ ]:
url = "https://www.fifa.com/es/tournaments/mens/worldcup/canadamexicousa2026/qualifiers/conmebol/scores-fixtures?country=CO&wtw-filter=ALL"
html = urlopen(url)
html

In [ ]:
import requests

In [ ]:
url = "https://www.fifa.com/es/tournaments/mens/worldcup/canadamexicousa2026/qualifiers/conmebol/scores-fixtures?country=CO&wtw-filter=ALL"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}
resp = requests.get(url, headers=headers, allow_redirects=True, timeout=30)
print(resp.status_code, resp.url)
print(resp.text[:800])


200 https://www.fifa.com/es/tournaments/mens/worldcup/canadamexicousa2026/qualifiers/conmebol/scores-fixtures?country=CO&wtw-filter=ALL
<!doctype html><html lang="en" dir="ltr"><head><meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/><meta name="theme-color" content="#020F2A"/><meta name="msapplication-TileImage" content="/mstile-150x150.png?v=6b9bed24d1df59ca113a3828909d3924"><meta name="msapplication-TileColor" content="#326295"/><meta name="google-site-verification" content="boCUJZPlju606sys-ZcnqZAThCMVJWEvwc6JXvElJTE"/><link rel="icon" href="/favicon.ico?v=4c4914f90c578869e7375b03cf029202"/><link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"/><link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png?v=1ea068c804e8ba88b84f6e9598e3172d"/><link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png?v


In [ ]:
resp.text

'<!doctype html><html lang="en" dir="ltr"><head><meta charset="utf-8"/><meta name="viewport" content="width=device-width,initial-scale=1"/><meta name="theme-color" content="#020F2A"/><meta name="msapplication-TileImage" content="/mstile-150x150.png?v=6b9bed24d1df59ca113a3828909d3924"><meta name="msapplication-TileColor" content="#326295"/><meta name="google-site-verification" content="boCUJZPlju606sys-ZcnqZAThCMVJWEvwc6JXvElJTE"/><link rel="icon" href="/favicon.ico?v=4c4914f90c578869e7375b03cf029202"/><link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"/><link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png?v=1ea068c804e8ba88b84f6e9598e3172d"/><link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png?v=7ee355e68b687435c3e5f74e2e831325"/><link rel="apple-touch-icon" href="/apple-touch-icon.png?v=a087933e3cf148cb71a96095c8aa2dac"/><link rel="manifest" href="/manifest.webmanifest?v=82b16ef736854febec

## Y ahora...

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.fifa.com/",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}

In [ ]:
api_url = "https://api.fifa.com/api/v3/teamform/43925?idCompetition=520&from=2023-09-07T00%3A00%3A00Z&to=2025-09-09T23%3A59%3A59Z&count=5&language=es"  # <-- la que veas en Network

In [ ]:
api_url = "https://api.fifa.com/api/v3/calendar/520/288315/288316/standing?language=es&count=200"

In [ ]:
r = requests.get(api_url, headers=headers, timeout=30)
r.raise_for_status()
data = r.json()
data

{'ContinuationToken': None,
 'ContinuationHash': None,
 'Results': [{'MatchDay': 18,
   'IdCompetition': '520',
   'IdSeason': '288315',
   'IdStage': '288316',
   'IdGroup': None,
   'IdTeam': '43922',
   'Date': '2023-09-07T22:30:00Z',
   'Group': [],
   'Won': 12,
   'Lost': 4,
   'Drawn': 2,
   'Played': 18,
   'HomeWon': 7,
   'HomeLost': 1,
   'HomeDrawn': 1,
   'HomePlayed': 9,
   'AwayWon': 5,
   'AwayLost': 3,
   'AwayDrawn': 1,
   'AwayPlayed': 9,
   'Against': 10,
   'For': 31,
   'HomeAgainst': 4,
   'HomeFor': 20,
   'AwayAgainst': 6,
   'AwayFor': 11,
   'Position': 1,
   'HomePosition': 1,
   'AwayPosition': 1,
   'Points': 38,
   'HomePoints': 22,
   'AwayPoints': 16,
   'PreviousPosition': 1,
   'GoalsDiference': 21,
   'TeamConductScore': 0,
   'QualificationStatus': None,
   'IsLive': False,
   'Team': {'IdTeam': '43922',
    'IdConfederation': 'CONMEBOL',
    'ActiveStatus': None,
    'Type': 1,
    'AgeType': 7,
    'FootballType': 0,
    'Gender': 1,
    'Name': [

In [ ]:
data.keys()

dict_keys(['ContinuationToken', 'ContinuationHash', 'Results'])

In [ ]:
api_url = "https://api.fifa.com/api/v3/calendar/520/288315/288316/standing?language=es&count=200"

r = requests.get(api_url, headers=headers, timeout=30)
r.raise_for_status()
data = r.json()
data

{'ContinuationToken': None,
 'ContinuationHash': None,
 'Results': [{'MatchDay': 18,
   'IdCompetition': '520',
   'IdSeason': '288315',
   'IdStage': '288316',
   'IdGroup': None,
   'IdTeam': '43922',
   'Date': '2023-09-07T22:30:00Z',
   'Group': [],
   'Won': 12,
   'Lost': 4,
   'Drawn': 2,
   'Played': 18,
   'HomeWon': 7,
   'HomeLost': 1,
   'HomeDrawn': 1,
   'HomePlayed': 9,
   'AwayWon': 5,
   'AwayLost': 3,
   'AwayDrawn': 1,
   'AwayPlayed': 9,
   'Against': 10,
   'For': 31,
   'HomeAgainst': 4,
   'HomeFor': 20,
   'AwayAgainst': 6,
   'AwayFor': 11,
   'Position': 1,
   'HomePosition': 1,
   'AwayPosition': 1,
   'Points': 38,
   'HomePoints': 22,
   'AwayPoints': 16,
   'PreviousPosition': 1,
   'GoalsDiference': 21,
   'TeamConductScore': 0,
   'QualificationStatus': None,
   'IsLive': False,
   'Team': {'IdTeam': '43922',
    'IdConfederation': 'CONMEBOL',
    'ActiveStatus': None,
    'Type': 1,
    'AgeType': 7,
    'FootballType': 0,
    'Gender': 1,
    'Name': [

In [ ]:
results = data["Results"]

# Extraemos los campos principales
rows = []
for r in results:
    rows.append({
        "Equipo": r["Team"]["Name"][0]["Description"],
        "Partidos": r["Played"],
        "Ganados": r["Won"],
        "Empatados": r["Drawn"],
        "Perdidos": r["Lost"],
        "Goles a favor": r["For"],
        "Goles en contra": r["Against"],
        "Diferencia": r["GoalsDiference"],
        "Puntos": r["Points"],
        "Posición": r["Position"]
    })

# Convertimos en DataFrame
df = pd.DataFrame(rows)

# Ordenamos por posición (por si acaso)
df = df.sort_values("Posición").reset_index(drop=True)

df


,Equipo,Partidos,Ganados,Empatados,Perdidos,Goles a favor,Goles en contra,Diferencia,Puntos,Posición
0,Argentina,18,12,2,4,31,10,21,38,1
1,Ecuador,18,8,8,2,14,5,9,29,2
2,Colombia,18,7,7,4,28,18,10,28,3
3,Uruguay,18,7,7,4,22,12,10,28,4
4,Brasil,18,8,4,6,24,17,7,28,5
5,Paraguay,18,7,7,4,14,10,4,28,6
6,Bolivia,18,6,2,10,17,35,-18,20,7
7,Venezuela,18,4,6,8,18,28,-10,18,8
8,Perú,18,2,6,10,6,21,-15,12,9
9,Chile,18,2,5,11,9,27,-18,11,10


# 📑 BeautifulSoup Cheat Sheet — Selectores CSS útiles

En **BeautifulSoup**, `soup.select("selector")` permite buscar elementos en el HTML usando sintaxis **CSS**.  
Aquí algunos de los más usados para **web scraping**:

---

## 🔹 Tablas
- `soup.select("table")` → todas las tablas `<table>`.
- `soup.select("table tr")` → todas las filas de todas las tablas.
- `soup.select("table#id_tabla")` → tabla con atributo `id="id_tabla"`.
- `soup.select("table.data tbody tr")` → filas de `<tbody>` en una tabla con clase `data`.
- `soup.select("tr td")` → todas las celdas `<td>` de todas las filas.

---

## 🔹 Enlaces
- `soup.select("a")` → todas las etiquetas de enlace `<a>`.
- `soup.select("a[href]")` → solo enlaces que tengan atributo `href`.
- `soup.select("a.external")` → enlaces con clase `external`.
- `soup.select("nav a")` → enlaces dentro de un bloque `<nav>`.

---

## 🔹 Imágenes
- `soup.select("img")` → todas las imágenes `<img>`.
- `soup.select("img[src]")` → imágenes con atributo `src`.
- `soup.select("img.thumbnail")` → imágenes con clase `thumbnail`.
- `soup.select("div.gallery img")` → imágenes dentro de un div con clase `gallery`.

---

## 🔹 Documentos (PDF, DOC, etc.)
- `soup.select("a[href$='.pdf']")` → enlaces a documentos PDF.
- `soup.select("a[href$='.docx']")` → enlaces a documentos Word.
- `soup.select("a[href*='download']")` → enlaces que contienen la palabra "download" en la URL.

---

## 🔹 Encabezados y texto
- `soup.select("h1, h2, h3")` → todos los títulos `<h1>`, `<h2>` y `<h3>`.
- `soup.select("p")` → todos los párrafos `<p>`.
- `soup.select("div.article p")` → párrafos dentro de un div con clase `article`.

---

## 🔹 Listas
- `soup.select("ul li")` → ítems de listas no ordenadas `<ul>`.
- `soup.select("ol li")` → ítems de listas ordenadas `<ol>`.

---

## 🔹 Formularios
- `soup.select("form")` → todos los formularios `<form>`.
- `soup.select("input")` → campos `<input>`.
- `soup.select("input[type='text']")` → campos de texto.
- `soup.select("button[type='submit']")` → botones de envío.

---

## 🧩 Tips adicionales
- `#id` → selecciona por id: `soup.select("#main")`.
- `.clase` → selecciona por clase: `soup.select(".content")`.
- `tag[attr='valor']` → por atributo: `soup.select("div[data-type='article']")`.
- `tag1 tag2` → descendientes: `soup.select("div p")` → párrafos dentro de div.
- `tag1 > tag2` → hijos directos: `soup.select("ul > li")`.

---
